# IHGAMP - Step 1: WSI Preprocessing

This notebook handles:
1. Building a manifest of all available WSI files
2. Extracting tiles from whole slide images
3. Quality control and tissue detection

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm

from src.preprocessing import TileExtractor

# ============================================================
# CONFIGURATION - UPDATE THESE PATHS
# ============================================================
WSI_ROOT = Path('/path/to/your/slides')
OUTPUT_DIR = Path('./outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def build_manifest(wsi_root: Path) -> pd.DataFrame:
    """Recursively find all WSI files and extract metadata."""
    WSI_EXTENSIONS = {'.svs', '.tif', '.tiff', '.ndpi', '.scn', '.mrxs'}
    rows = []
    for ext in WSI_EXTENSIONS:
        for f in wsi_root.rglob(f'*{ext}'):
            name = f.stem
            parts = name.split('-')
            patient = '-'.join(parts[:3]) if len(parts) >= 3 and parts[0] == 'TCGA' else name
            rows.append({'slide_path': str(f), 'slide_name': name, 'patient': patient})
    return pd.DataFrame(rows)

manifest = build_manifest(WSI_ROOT)
print(f'Found {len(manifest)} slides from {manifest["patient"].nunique()} patients')

In [ ]:
# Initialize tile extractor
extractor = TileExtractor(
    patch_size=256,
    stride=256,
    tissue_threshold=0.60,
    patch_cap=3000
)

# Process slides
results = []
for _, row in tqdm(manifest.iterrows(), total=len(manifest)):
    result = extractor.process_slide(row['slide_path'])
    results.append(result)

print(f'Processed {len(results)} slides')

In [ ]:
# Save manifest
manifest.to_csv(OUTPUT_DIR / 'slide_manifest.csv', index=False)
print(f'Saved to: {OUTPUT_DIR / "slide_manifest.csv"}')